# Modelo de Classificação: Cats vs Dogs com ResNet (transfer learning)

* Mesmo problema e mesma base do `cats_and_dogs_224.ipynb`, mas aqui usamos uma **ResNet18 já treinada na ImageNet** e ajustamos (fine-tuning) para separar gatos e cachorros.
* A limpeza dos dados e a divisão entre treino, validação e teste são as mesmas dos outros notebooks: imagens idênticas são removidas e imagens quase idênticas ficam sempre do mesmo lado da divisão.
* O data augmentation é aleatório a cada época e só vale para o treino. Validação e teste ficam só com imagens originais.
* O conjunto de teste só é aberto no final.

## 1. Configuraçaõ Inicial

In [ ]:
%pip install kagglehub torch torchvision numpy pillow scipy matplotlib

In [ ]:
import hashlib
import json
import os
from collections import Counter, defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from PIL import Image
from torch.utils.data import DataLoader, Subset, TensorDataset

SEED = 18
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)

In [ ]:
import kagglehub

caminho = kagglehub.dataset_download("bhavikjikadara/dog-and-cat-classification-dataset")
print(caminho)

## 2. Exploração dos Dados

In [ ]:
for pasta, subpastas, arquivos in os.walk(caminho):
    print(pasta, "-", len(arquivos), "arquivos")

In [ ]:
info = []          # um item por arquivo que abriu como imagem
nao_abriram = []   # arquivos que não abriram como imagem

for pasta, _, nomes in os.walk(caminho):
    for nome in nomes:
        arquivo = os.path.join(pasta, nome)
        try:
            with Image.open(arquivo) as img:
                img.load()
                largura, altura = img.size
                modo = img.mode
        except Exception:
            nao_abriram.append(arquivo)
            continue
        with open(arquivo, "rb") as f:
            impressao = hashlib.md5(f.read()).hexdigest()
        info.append({
            "arquivo": arquivo,
            "classe": os.path.basename(pasta),
            "largura": largura,
            "altura": altura,
            "modo": modo,
            "hash": impressao,
        })

print("Imagens que abriram:", len(info))
print("Arquivos que não abriram:", len(nao_abriram), nao_abriram[:10])
print("Pastas (classes):", Counter(d["classe"] for d in info))
print("Modos de cor:", Counter(d["modo"] for d in info))

* Para observar alguns erros de classificação ou se existem arquivos idênticos (e retirá-las posteriormente)

In [ ]:
por_hash = defaultdict(list)
for d in info:
    por_hash[d["hash"]].append(d)

grupos_identicos = [g for g in por_hash.values() if len(g) > 1]
em_classes_diferentes = [g for g in grupos_identicos if len({d["classe"] for d in g}) > 1]

print("Grupos de arquivos idênticos:", len(grupos_identicos))
print("Desses, em classes diferentes:", len(em_classes_diferentes))
for g in em_classes_diferentes[:10]:
    print([d["arquivo"] for d in g])

In [ ]:
# Grupos com classes diferentes saem inteiros; nos outros, fica um arquivo só.
imagens = [g[0] for g in por_hash.values() if len({d["classe"] for d in g}) == 1]

def impressao_visual(arquivo):
    # Versão 9 x 8 em cinza: cada pixel é comparado com o vizinho da direita.
    with Image.open(arquivo) as im:
        cinza = np.asarray(im.convert("L").resize((9, 8), Image.BILINEAR), dtype=np.int16)
    return (cinza[:, 1:] > cinza[:, :-1]).tobytes()

# Imagens quase idênticas ficam com a mesma origem e vão para o mesmo lado na divisão.
origens = {}
for d in imagens:
    d["origem"] = origens.setdefault(impressao_visual(d["arquivo"]), len(origens))

print("Imagens depois de tirar as repetidas:", len(imagens))
print("Origens diferentes:", len(origens))

* Padronizar tamanho das imagens: 224 x 224, mantendo a proporção (lado menor vira 224 e o excesso do lado maior é cortado no centro) (com interpolação bilinear)

In [ ]:
# Mesmo filtro do notebook original: imagens com algum lado menor que 94 px são descartadas.
LADO_MINIMO = 94
imagens = [d for d in imagens if min(d["largura"], d["altura"]) >= LADO_MINIMO]
print("Imagens com pelo menos", LADO_MINIMO, "px em cada lado:", len(imagens))

LARGURA, ALTURA = 224, 224
ampliadas = sum(1 for d in imagens if min(d["largura"], d["altura"]) < 224)
print("Tamanho-alvo:", LARGURA, "x", ALTURA, "(largura x altura)")
print("Imagens que vão ser ampliadas (lado menor < 224 px):", ampliadas)

In [ ]:
def redimensionar(img):
    # O lado menor da imagem vira 224 e o lado maior é cortado no centro. A proporção se mantém.
    # Interpolação bilinear do PIL: ao reduzir, ela faz a média da área coberta por cada pixel novo
    # (evita o serrilhado de simplesmente pegar um pixel vizinho).
    h, w = img.shape[:2]
    escala = max(ALTURA / h, LARGURA / w)
    corte_h, corte_w = ALTURA / escala, LARGURA / escala
    esquerda, topo = (w - corte_w) / 2, (h - corte_h) / 2
    pil = Image.fromarray(np.clip(np.rint(img), 0, 255).astype(np.uint8))
    nova = pil.resize((LARGURA, ALTURA), Image.BILINEAR,
                      box=(esquerda, topo, esquerda + corte_w, topo + corte_h))
    return np.asarray(nova, dtype=np.float32)

## 3. Divisão entre Treino, Validação e Teste

* A divisão é feita **antes** do data augmentation, só com as imagens originais. Assim, nada que vem de uma imagem de validação ou de teste pode aparecer no treino.
* Imagens quase idênticas (mesma `origem`) ficam sempre do mesmo lado.

In [ ]:
grupo = np.array([d["origem"] for d in imagens])

def separar_por_grupo(indices, fracao):
    # Sorteia grupos inteiros até juntar a fração pedida das imagens.
    contagem = Counter(grupo[indices].tolist())
    alvo = round(fracao * len(indices))
    escolhidos, total = [], 0
    for g in rng.permutation(sorted(contagem)):
        if total >= alvo:
            break
        escolhidos.append(g)
        total += contagem[g]
    separado = np.isin(grupo[indices], escolhidos)
    return indices[~separado], indices[separado]

# Posições na lista `imagens` (ainda sem nenhuma imagem aumentada)
treino_original, teste = separar_por_grupo(np.arange(len(imagens)), 0.15)
treino_original, validacao = separar_por_grupo(treino_original, 0.15)

print("Treino:", len(treino_original), "| validação:", len(validacao), "| teste:", len(teste))

## 4. Carregamento das imagens

* Só as imagens originais, sem cópias aumentadas: o augmentation é feito na hora, a cada lote. Ficam guardadas como `uint8` (0 a 255); a conversão para `float` (0 a 1) é feita por lote.

In [ ]:
classes = sorted({d["classe"] for d in imagens})
n = len(imagens)

X = np.empty((n, ALTURA, LARGURA, 3), dtype=np.uint8)   # (n, altura, largura, 3), de 0 a 255
y = np.empty(n, dtype=np.int64)

for i, d in enumerate(imagens):
    with Image.open(d["arquivo"]) as im:
        img = np.asarray(im.convert("RGB"), dtype=np.float32)   # valores de 0 a 255
    X[i] = np.clip(np.rint(redimensionar(img)), 0, 255).astype(np.uint8)
    y[i] = classes.index(d["classe"])

X = torch.from_numpy(X)
y = torch.from_numpy(y)
dados = TensorDataset(X, y)

treino = treino_original   # só imagens originais; o augmentation vem no treino
print("Classes:", classes)
print("Formato:", tuple(X.shape), "| tipo:", X.dtype)
print("Treino:", len(treino), "| validação:", len(validacao), "| teste:", len(teste))

In [ ]:
dispositivo = "cuda" if torch.cuda.is_available() else "cpu"

def preparar(xb):
    # uint8 (lote, altura, largura, 3) -> float (lote, 3, altura, largura), de 0 a 1
    return xb.to(dispositivo).permute(0, 3, 1, 2).float().div(255).contiguous()

def avaliar(indices):
    # Devolve (perda, acurácia) nas imagens pedidas, com dropout desligado.
    modelo.eval()
    acertos, soma_perda = 0, 0.0
    with torch.no_grad():
        for xb, yb in DataLoader(Subset(dados, indices), batch_size=32):
            z = modelo.antes_da_softmax(preparar(xb))
            yb = yb.to(dispositivo)
            soma_perda += F.cross_entropy(z, yb, reduction="sum").item()
            acertos += (z.argmax(dim=1) == yb).sum().item()
    modelo.train()
    return soma_perda / len(indices), acertos / len(indices)

def acuracia(indices):
    return avaliar(indices)[1]

## 5. Data Augmentation aleatório a cada época

* Aplicado por lote, na hora do treino, só nas imagens de treino: espelhamento horizontal (50%), zoom (70% a 100% da imagem) com deslocamento, rotação de -15° a +15° (bordas preenchidas espelhando a imagem) e brilho e contraste entre 0,8 e 1,2.

In [ ]:
import math

def aumentar(x):
    # x: float (lote, 3, altura, largura), de 0 a 1. Cada imagem do lote recebe sorteios próprios.
    B = x.shape[0]
    sorteio = lambda a, b: torch.empty(B, device=x.device).uniform_(a, b)

    zoom = sorteio(0.7, 1.0)                                  # fração da imagem que fica visível
    angulo = sorteio(-15, 15) * math.pi / 180
    espelho = torch.where(torch.rand(B, device=x.device) < 0.5, -1.0, 1.0)
    tx = sorteio(-1, 1) * (1 - zoom)                          # deslocamento, sem sair da imagem
    ty = sorteio(-1, 1) * (1 - zoom)
    cos, sen = torch.cos(angulo), torch.sin(angulo)

    # Matriz 2 x 3 de cada imagem: espelha, dá zoom, gira e desloca de uma vez só
    teta = torch.stack([
        torch.stack([zoom * cos * espelho, -zoom * sen, tx], dim=1),
        torch.stack([zoom * sen * espelho, zoom * cos, ty], dim=1),
    ], dim=1)
    grade = F.affine_grid(teta, x.shape, align_corners=False)
    x = F.grid_sample(x, grade, mode="bilinear", padding_mode="reflection", align_corners=False)

    # Brilho e contraste
    media = x.mean(dim=(1, 2, 3), keepdim=True)
    brilho = sorteio(0.8, 1.2).view(B, 1, 1, 1)
    contraste = sorteio(0.8, 1.2).view(B, 1, 1, 1)
    return ((x - media) * contraste + media * brilho).clamp(0, 1)

In [ ]:
import matplotlib.pyplot as plt

# A primeira imagem é uma imagem de treino original; as outras 5 são sorteios diferentes do augmentation
original = preparar(X[treino_original[:1]])
fig, eixos = plt.subplots(1, 6, figsize=(14, 2.6))
eixos[0].imshow(original[0].permute(1, 2, 0).cpu().numpy())
eixos[0].set_title("original")
for ax in eixos[1:]:
    ax.imshow(aumentar(original)[0].permute(1, 2, 0).cpu().numpy())
    ax.set_title("aumentada")
for ax in eixos:
    ax.axis("off")
plt.show()

## 6. Modelo

* **ResNet18 com pesos da ImageNet** (`torchvision`). A última camada (1000 classes) é trocada por uma de 2 saídas, gato e cachorro. **Todas as camadas são treinadas** (fine-tuning), com taxa de aprendizado pequena para não destruir o que a rede já aprendeu.
* A normalização da ImageNet (média e desvio por canal) fica dentro do modelo, porque os pesos foram treinados com ela. O resto do notebook continua trabalhando com imagens de 0 a 1.
* Os pesos são baixados na primeira execução (~45 MB).

In [ ]:
MEDIA = (0.485, 0.456, 0.406)    # estatísticas da ImageNet
DESVIO = (0.229, 0.224, 0.225)

class ResNetGatosCachorros(nn.Module):
    def __init__(self):
        super().__init__()
        self.rede = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1)
        self.rede.fc = nn.Linear(self.rede.fc.in_features, 2)
        self.register_buffer("media", torch.tensor(MEDIA).view(1, 3, 1, 1))
        self.register_buffer("desvio", torch.tensor(DESVIO).view(1, 3, 1, 1))

    def antes_da_softmax(self, x):
        # x: imagens de 0 a 1
        return self.rede((x - self.media) / self.desvio)

    def forward(self, x):
        return torch.softmax(self.antes_da_softmax(x), dim=1)

## 7. Treino e Validação dos Dados

In [ ]:
EPOCAS = 5   # com pesos pré-treinados, poucas épocas bastam

torch.manual_seed(SEED)
modelo = ResNetGatosCachorros().to(dispositivo)
otimizador = torch.optim.Adam(modelo.parameters(), lr=1e-4)
lotes_treino = DataLoader(Subset(dados, treino), batch_size=32, shuffle=True,
                          generator=torch.Generator().manual_seed(SEED))

historico = {"perda_treino": [], "acc_treino": [], "perda_val": [], "acc_val": []}
melhor_acuracia = 0

for epoca in range(1, EPOCAS + 1):
    modelo.train()
    soma_perda, acertos, vistos = 0.0, 0, 0
    for xb, yb in lotes_treino:
        xb, yb = aumentar(preparar(xb)), yb.to(dispositivo)   # augmentation novo a cada lote, a cada época
        z = modelo.antes_da_softmax(xb)
        perda = F.cross_entropy(z, yb)
        otimizador.zero_grad()
        perda.backward()
        otimizador.step()
        # Perda e acurácia de treino: média durante a época, em imagens aumentadas.
        soma_perda += perda.item() * len(yb)
        acertos += (z.argmax(dim=1) == yb).sum().item()
        vistos += len(yb)

    perda_val, acc = avaliar(validacao)   # validação: só imagens originais
    historico["perda_treino"].append(soma_perda / vistos)
    historico["acc_treino"].append(acertos / vistos)
    historico["perda_val"].append(perda_val)
    historico["acc_val"].append(acc)
    json.dump(historico, open("historico_treino_resnet.json", "w"))
    print(f"Época {epoca:3d}   perda no treino: {soma_perda / vistos:.4f}   acurácia no treino: {acertos / vistos:.3f}   "
          f"perda na validação: {perda_val:.4f}   acurácia na validação: {acc:.3f}")

    if acc > melhor_acuracia:
        melhor_acuracia, melhor_epoca = acc, epoca
        melhores_pesos = {k: v.cpu().clone() for k, v in modelo.state_dict().items()}
        torch.save(melhores_pesos, "modelo_cats_and_dogs_resnet.pt")

# O modelo final é o da época com a melhor acurácia na validação.
modelo.load_state_dict(melhores_pesos)
print(f"Melhor época: {melhor_epoca} | acurácia na validação: {melhor_acuracia:.3f}")

## 8. Análise dos Resultados

#### Gráficos Acurácia e Perda no Treino

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# Usa o modelo fechado do treino da ResNet. Nada aqui treina de novo.
modelo.load_state_dict(torch.load("modelo_cats_and_dogs_resnet.pt", map_location=dispositivo))
modelo.eval()

AZUL, LARANJA, CINZA = "#2a78d6", "#eb6834", "#8a8985"
TINTA, TINTA_2, GRADE = "#0b0b0b", "#52514e", "#e6e5e0"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "text.color": TINTA, "axes.labelcolor": TINTA_2, "xtick.color": TINTA_2, "ytick.color": TINTA_2,
    "axes.edgecolor": GRADE, "axes.grid": True, "grid.color": GRADE, "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 11,
    "axes.titlesize": 13, "axes.titleweight": "bold", "axes.titlelocation": "left",
})

# Perda e acurácia de treino: média durante a época, com dropout ligado, em imagens aumentadas.
hist = json.load(open("historico_treino_resnet.json"))
epocas = np.arange(1, len(hist["acc_val"]) + 1)
ep = int(np.argmax(hist["acc_val"])) + 1   # melhor época na validação (a primeira, se houver empate)

fig, (ax_acc, ax_perda) = plt.subplots(1, 2, figsize=(13, 4.8))

ax_acc.axhline(0.5, color=TINTA_2, lw=1, ls=":")
ax_acc.text(epocas[-1], 0.51, "acaso (50%)", color=TINTA_2, fontsize=9, ha="right")
ax_acc.plot(epocas, hist["acc_treino"], color=AZUL, lw=2, label="Treino")
ax_acc.plot(epocas, hist["acc_val"], color=LARANJA, lw=2, label="Validação")
ax_acc.plot(ep, hist["acc_val"][ep - 1], "o", ms=9, color=TINTA, mec="#fcfcfb", mew=2, zorder=5)
ax_acc.annotate(f"Modelo fechado\nÉpoca {ep}: {hist['acc_val'][ep - 1]:.1%}",
                (ep, hist["acc_val"][ep - 1]), xytext=(epocas[-1] * 0.45, 0.62), color=TINTA, fontsize=10,
                arrowprops=dict(arrowstyle="-", color=TINTA_2, lw=1))
ax_acc.set(title="Acurácia por época", xlabel="Época", ylabel="Acurácia", xlim=(1, epocas[-1]), ylim=(0.45, 1.0))
ax_acc.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax_acc.legend(frameon=False, loc="lower right", bbox_to_anchor=(1, 0.08), fontsize=9.5)

ax_perda.plot(epocas, hist["perda_treino"], color=AZUL, lw=2, label="Treino")
ax_perda.plot(epocas, hist["perda_val"], color=LARANJA, lw=2, label="Validação")
ax_perda.set(title="Perda (entropia cruzada) por época", xlabel="Época", ylabel="Perda", xlim=(1, epocas[-1]), ylim=(0, None))
ax_perda.legend(frameon=False, loc="upper right", fontsize=9.5)

fig.tight_layout()
plt.show()

## 9. Avaliação final no Teste

* O teste só é aberto agora, no final, e não foi usado em nenhuma decisão.

In [ ]:
print("Acurácia no teste:", acuracia(teste))

#### Matriz de Confusão 

In [ ]:
NOMES = {"Cat": "gato", "Dog": "cachorro"}
rotulos = [NOMES[c] for c in classes]

# Previsões do modelo fechado no conjunto de teste
probs = []
with torch.no_grad():
    for xb, _ in DataLoader(Subset(dados, teste), batch_size=32):
        probs.append(modelo(preparar(xb)).cpu())
probs = torch.cat(probs)
previsto = probs.argmax(dim=1)
real = y[teste]

matriz = np.zeros((2, 2), dtype=int)
for r_, p_ in zip(real.tolist(), previsto.tolist()):
    matriz[r_, p_] += 1

print(f"Acertos: {matriz.trace()} de {matriz.sum()} ({matriz.trace() / matriz.sum():.4f})")
for i, nome in enumerate(rotulos):
    print(f"{nome:9s} recall {matriz[i, i] / matriz[i].sum():.1%}   precisão {matriz[i, i] / matriz[:, i].sum():.1%}")

azuis = LinearSegmentedColormap.from_list("azuis", ["#cde2fb", "#6da7ec", "#2a78d6", "#104281"])
fig, ax = plt.subplots(figsize=(5.8, 5))
ax.imshow(matriz, cmap=azuis, vmin=0)
for i in range(2):
    for j in range(2):
        cor = "white" if matriz[i, j] > matriz.max() / 2 else TINTA
        ax.text(j, i - 0.07, f"{matriz[i, j]}", ha="center", va="center", color=cor, fontsize=22, fontweight="bold")
        ax.text(j, i + 0.13, f"{matriz[i, j] / matriz[i].sum():.1%} dos {rotulos[i]}s", ha="center", va="center", color=cor, fontsize=10)
ax.set(xticks=[0, 1], yticks=[0, 1], xticklabels=rotulos, yticklabels=rotulos,
       xlabel="Classe prevista", ylabel="Classe real", title="Matriz de confusão no teste")
ax.grid(False)
ax.tick_params(length=0)
for lado in ax.spines.values():
    lado.set_visible(False)
ax.set_xticks([0.5], minor=True)
ax.set_yticks([0.5], minor=True)
ax.grid(which="minor", color="#fcfcfb", lw=3)
ax.tick_params(which="minor", length=0)
fig.tight_layout()
plt.show()

#### Exemplos de imagens que foram classificadas errado

In [ ]:
def mostrar_erros(classe_real, quantos=12):
    classe_prevista = 1 - classe_real
    erros = ((real == classe_real) & (previsto == classe_prevista)).nonzero().ravel()
    # Do mais confiante para o menos; uma imagem por origem, para não repetir a mesma foto.
    erros = erros[probs[erros, classe_prevista].argsort(descending=True)]
    vistos, escolhidos = set(), []
    for e in erros.tolist():
        origem = int(grupo[teste[e]])
        if origem not in vistos:
            vistos.add(origem)
            escolhidos.append(e)
        if len(escolhidos) == quantos:
            break

    colunas = 6
    linhas = int(np.ceil(len(escolhidos) / colunas))
    fig, eixos = plt.subplots(linhas, colunas, figsize=(13, 2.5 * linhas), squeeze=False)
    for ax in eixos.ravel():
        ax.axis("off")
    for ax, e in zip(eixos.ravel(), escolhidos):
        ax.imshow(X[teste[e]].numpy(), interpolation="nearest")
        ax.set_title(f"{probs[e, classe_prevista]:.1%} de certeza", fontsize=10, fontweight="normal", color=TINTA_2, loc="center")
    fig.suptitle(f"Era {rotulos[classe_real]}, previu {rotulos[classe_prevista]}: "
                 f"{len(erros)} erros no teste. Estes são os {len(escolhidos)} com maior certeza.",
                 x=0.01, ha="left", fontsize=13, fontweight="bold")
    fig.tight_layout()
    plt.show()

mostrar_erros(classes.index("Cat"))
mostrar_erros(classes.index("Dog"))